# Projeto completo de qualidade de vinhos

**Disciplina 2 de Aprendizado de Máquina — IASEG**

Este notebook reúne todos os passos exigidos no enunciado e incorpora os conteúdos publicados depois dos dois primeiros notebooks: árvore e ajuste de complexidade, validação cruzada, floresta aleatória, comparação justa, auditoria de vazamento e o banco de perguntas de 29/09/2026.

**Pergunta:** é possível estimar a nota `quality` de uma amostra usando 11 medidas físico-químicas e o tipo do vinho?  
**Saída:** um número, portanto a tarefa é de regressão.  
**Uso proposto:** ordenar amostras para uma segunda degustação humana. O modelo não aprova lotes sozinho.

O fluxo preserva 20% dos dados limpos como teste final. Todas as escolhas de modelo e do corte operacional usam somente os 80% de desenvolvimento. Execute as células em ordem a partir da raiz do repositório ou da pasta `projeto`.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import (
    KFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEMENTE = 42
ALVO = "quality"
PASTA_ATUAL = Path.cwd()
PASTA_PROJETO = PASTA_ATUAL if PASTA_ATUAL.name == "projeto" else PASTA_ATUAL / "projeto"
PASTA_DADOS = PASTA_PROJETO / "dados"
PASTA_RESULTADOS = PASTA_PROJETO / "resultados"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

print("Python pronto. pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)


## Passo 1 — Definir o problema e auditar os dados

Os arquivos públicos do UCI Wine Quality contêm 1.599 vinhos tintos e 4.898 brancos. Cada arquivo possui 11 medidas físico-químicas e a nota sensorial. Acrescentamos `wine_type` antes de unir as tabelas.

A auditoria procura ausências, valores inválidos, duplicatas completas e entradas iguais com notas diferentes. O alvo fica fora de `X` para impedir vazamento.


In [ ]:
arquivos = {
    "red": PASTA_DADOS / "winequality-red.csv",
    "white": PASTA_DADOS / "winequality-white.csv",
}

tabelas = {}
for tipo, caminho in arquivos.items():
    tabela = pd.read_csv(caminho, sep=";")
    tabela = tabela.apply(pd.to_numeric, errors="raise")
    if tabela.empty or tabela.isna().any().any() or not np.isfinite(tabela.to_numpy()).all():
        raise ValueError(f"Arquivo inválido: {caminho}")
    tabelas[tipo] = tabela

dados_brutos = pd.concat(
    [
        tabelas["red"].assign(wine_type="red"),
        tabelas["white"].assign(wine_type="white"),
    ],
    ignore_index=True,
)
COLUNAS_ENTRADA = [c for c in dados_brutos.columns if c != ALVO]
duplicatas_completas = int(dados_brutos.duplicated().sum())
perfis_conflitantes = int(
    dados_brutos.groupby(COLUNAS_ENTRADA, dropna=False)[ALVO].nunique().gt(1).sum()
)

auditoria = pd.DataFrame(
    {
        "item": [
            "linhas originais", "colunas após wine_type", "células ausentes",
            "duplicatas completas", "perfis iguais com notas diferentes",
            "nota mínima", "nota máxima", "percentual de notas 5 ou 6",
        ],
        "resultado": [
            len(dados_brutos), dados_brutos.shape[1], int(dados_brutos.isna().sum().sum()),
            duplicatas_completas, perfis_conflitantes, dados_brutos[ALVO].min(),
            dados_brutos[ALVO].max(),
            f"{100 * dados_brutos[ALVO].isin([5, 6]).mean():.1f}%",
        ],
    }
)
display(auditoria)
display(dados_brutos.head(3))


In [ ]:
distribuicao = dados_brutos[ALVO].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7.4, 3.8), constrained_layout=True)
ax.bar(distribuicao.index.astype(str), distribuicao.values, color="#7D263C")
ax.set(title="Distribuição das notas na base original", xlabel="Nota quality", ylabel="Número de amostras")
for x, valor in enumerate(distribuicao.values):
    ax.text(x, valor + 35, str(valor), ha="center", fontsize=8)
fig.savefig(PASTA_FIGURAS / "distribuicao_notas.png", bbox_inches="tight")
plt.show()


## Passo 2 — Limpar e reservar o teste

Removemos somente linhas integralmente idênticas. Essa decisão reduz o risco de uma cópia aparecer no treino e outra no teste, mas não prova que toda repetição era erro. A base não possui identificador de amostra.

Reservamos 20% para o teste final com `random_state=42` e `stratify=y`. As cinco dobras posteriores usam apenas o conjunto de desenvolvimento. Para regressão, usamos `KFold` com embaralhamento e a mesma semente. O pipeline aprende codificação e escala dentro de cada dobra.


In [ ]:
base = dados_brutos.drop_duplicates(keep="first").copy()
X = base.drop(columns=ALVO)
y = base[ALVO]

X_desenvolvimento, X_teste, y_desenvolvimento, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)

assert len(base) == 5320
assert len(X_desenvolvimento) == 4256 and len(X_teste) == 1064
assert set(X_desenvolvimento.index).isdisjoint(X_teste.index)

divisao = pd.DataFrame(
    {
        "conjunto": ["Desenvolvimento", "Teste final"],
        "linhas": [len(X_desenvolvimento), len(X_teste)],
        "uso": ["seleção, ajuste e corte", "avaliação final uma vez"],
    }
)
display(divisao)
print("Linhas removidas como duplicatas completas:", len(dados_brutos) - len(base))


## Passo 3 — Definir a referência, a métrica e as dobras

O modelo de referência prevê sempre a mediana aprendida no treino. A métrica principal é o erro absoluto médio, MAE, porque mantém a unidade de pontos de nota. RMSE e R² complementam a leitura.

Os modelos recebem as mesmas cinco dobras. Regressão linear e k-NN padronizam as medidas dentro do pipeline. Árvores não precisam de escala, mas continuam dentro de um pipeline para manter a codificação consistente de `wine_type`.


In [ ]:
COLUNAS_NUMERICAS = [c for c in X.columns if c != "wine_type"]
COLUNAS_CATEGORICAS = ["wine_type"]

def criar_preparacao(padronizar=True):
    return ColumnTransformer(
        [
            ("numericas", StandardScaler() if padronizar else "passthrough", COLUNAS_NUMERICAS),
            (
                "tipo",
                OneHotEncoder(
                    categories=[["red", "white"]], drop="first",
                    sparse_output=False, handle_unknown="error",
                ),
                COLUNAS_CATEGORICAS,
            ),
        ]
    )

def criar_pipeline(modelo, padronizar=True):
    return Pipeline([("preparacao", criar_preparacao(padronizar)), ("modelo", modelo)])

dobras = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)
metricas_cv = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2",
}

def avaliar_validacao(nome, modelo):
    resultado = cross_validate(
        modelo, X_desenvolvimento, y_desenvolvimento,
        cv=dobras, scoring=metricas_cv, return_train_score=True, n_jobs=1,
    )
    erros = -resultado["test_mae"]
    return {
        "modelo": nome,
        "MAE treino": -resultado["train_mae"].mean(),
        "MAE VC": erros.mean(),
        "desvio MAE": erros.std(),
        "pior dobra": erros.max(),
        "melhor dobra": erros.min(),
        "RMSE VC": -resultado["test_rmse"].mean(),
        "R² VC": resultado["test_r2"].mean(),
    }


## Passo 4 — Ajustar a complexidade sem consultar o teste

Testamos profundidades de 1 a 18. A curva compara o MAE do treino com o MAE das partes de validação. O ponto de menor validação define a árvore. Depois dele, o treino continua melhorando enquanto a validação piora, sinal de sobreajuste.

O k-NN usa pesos pela distância e escolhe `k` nas mesmas dobras. Para a floresta, comparamos 100 e 350 árvores mantendo `max_features=0.7`, configuração do gabarito. O teste final não participa dessas escolhas.


In [ ]:
linhas_arvore = []
for profundidade in range(1, 19):
    linha = avaliar_validacao(
        f"profundidade {profundidade}",
        criar_pipeline(
            DecisionTreeRegressor(max_depth=profundidade, random_state=SEMENTE),
            padronizar=False,
        ),
    )
    linha["profundidade"] = profundidade
    linhas_arvore.append(linha)

curva_arvore = pd.DataFrame(linhas_arvore)
profundidade_escolhida = int(curva_arvore.loc[curva_arvore["MAE VC"].idxmin(), "profundidade"])
display(curva_arvore[["profundidade", "MAE treino", "MAE VC"]].round(3))
print("Profundidade escolhida pela validação:", profundidade_escolhida)

fig, ax = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
ax.plot(curva_arvore["profundidade"], curva_arvore["MAE treino"], "o-", label="Treino", color="#9B3046")
ax.plot(curva_arvore["profundidade"], curva_arvore["MAE VC"], "o-", label="Validação cruzada", color="#2B5B78")
ax.axvline(profundidade_escolhida, color="#B8862B", linestyle="--", label=f"Escolha: {profundidade_escolhida}")
ax.set(xticks=range(1, 19), xlabel="Profundidade máxima", ylabel="MAE", title="Complexidade da árvore")
ax.legend()
fig.savefig(PASTA_FIGURAS / "curva_arvore.png", bbox_inches="tight")
plt.show()


In [ ]:
ajustes = []
for k in [5, 10, 15, 25, 35]:
    linha = avaliar_validacao(
        f"k-NN k={k}",
        criar_pipeline(KNeighborsRegressor(n_neighbors=k, weights="distance")),
    )
    ajustes.append({"família": "k-NN", "configuração": f"k={k}, pesos=distância", "MAE VC": linha["MAE VC"]})

for n_arvores in [100, 350]:
    linha = avaliar_validacao(
        f"Floresta {n_arvores}",
        criar_pipeline(
            RandomForestRegressor(
                n_estimators=n_arvores, max_features=0.7,
                random_state=SEMENTE, n_jobs=1,
            ),
            padronizar=False,
        ),
    )
    ajustes.append({"família": "Floresta", "configuração": f"{n_arvores} árvores, max_features=0.7", "MAE VC": linha["MAE VC"]})

tabela_ajustes = pd.DataFrame(ajustes)
display(tabela_ajustes.round(3))


## Passo 5 — Comparar os modelos e congelar a escolha

Comparamos a referência, regressão linear, k-NN, árvore e floresta usando as mesmas dobras e o mesmo MAE. A floresta com 350 árvores teve o menor MAE médio de validação e foi escolhida antes de qualquer métrica de teste.

O MAE de treino quase nulo do k-NN com pesos por distância e o MAE muito baixo da floresta não justificam a escolha. Modelos flexíveis podem ajustar os exemplos conhecidos; a validação é o critério de generalização.


In [ ]:
modelos = {
    "Referência (mediana)": criar_pipeline(DummyRegressor(strategy="median"), padronizar=False),
    "Regressão linear": criar_pipeline(LinearRegression()),
    "k-NN": criar_pipeline(KNeighborsRegressor(n_neighbors=15, weights="distance")),
    "Árvore": criar_pipeline(
        DecisionTreeRegressor(max_depth=profundidade_escolhida, random_state=SEMENTE),
        padronizar=False,
    ),
    "Floresta aleatória": criar_pipeline(
        RandomForestRegressor(
            n_estimators=350, max_features=0.7,
            random_state=SEMENTE, n_jobs=1,
        ),
        padronizar=False,
    ),
}

comparacao_validacao = pd.DataFrame(
    [avaliar_validacao(nome, modelo) for nome, modelo in modelos.items()]
).sort_values("MAE VC")
display(comparacao_validacao.round(3))

MODELO_ESCOLHIDO = "Floresta aleatória"
print("Escolha congelada antes do teste:", MODELO_ESCOLHIDO)
print("Motivo: menor MAE médio nas mesmas cinco dobras; o ganho compensa a perda de legibilidade para a triagem proposta.")

fig, ax = plt.subplots(figsize=(7.8, 4.0), constrained_layout=True)
ordem = comparacao_validacao.sort_values("MAE VC", ascending=True)
ax.barh(ordem["modelo"], ordem["MAE VC"], color=["#2F6B56" if n == MODELO_ESCOLHIDO else "#9AA5AF" for n in ordem["modelo"]])
ax.set(xlabel="MAE médio na validação cruzada", title="Comparação antes do teste")
for y_pos, valor in enumerate(ordem["MAE VC"]):
    ax.text(valor + 0.003, y_pos, f"{valor:.3f}", va="center", fontsize=9)
fig.savefig(PASTA_FIGURAS / "comparacao_validacao.png", bbox_inches="tight")
plt.show()


### Regra operacional definida ainda no desenvolvimento

Chamamos de caso de interesse uma amostra com nota real `quality >= 7`. A política envia para degustação prioritária as amostras cuja nota prevista alcança o corte. As previsões fora da dobra simulam casos não usados no ajuste.

Escolhemos o maior corte em décimos que preserva pelo menos 75% de sensibilidade no desenvolvimento. Essa regra favorece não perder bons candidatos, aceitando falsos positivos. O corte é uma decisão operacional derivada do regressor.


In [ ]:
modelo_selecionado = modelos[MODELO_ESCOLHIDO]
previsoes_fora_dobra = cross_val_predict(
    modelo_selecionado, X_desenvolvimento, y_desenvolvimento,
    cv=dobras, n_jobs=1,
)
interesse_desenvolvimento = y_desenvolvimento.to_numpy() >= 7

linhas_corte = []
for corte in np.round(np.arange(5.0, 7.51, 0.1), 1):
    selecionados = previsoes_fora_dobra >= corte
    linhas_corte.append(
        {
            "corte": corte,
            "sensibilidade": recall_score(interesse_desenvolvimento, selecionados, zero_division=0),
            "precisão": precision_score(interesse_desenvolvimento, selecionados, zero_division=0),
            "F1": f1_score(interesse_desenvolvimento, selecionados, zero_division=0),
            "selecionados": int(selecionados.sum()),
        }
    )

curva_corte = pd.DataFrame(linhas_corte)
elegiveis = curva_corte[curva_corte["sensibilidade"] >= 0.75]
CORTE_ESCOLHIDO = float(elegiveis["corte"].max())
linha_corte = curva_corte.loc[curva_corte["corte"].eq(CORTE_ESCOLHIDO)].iloc[0]
display(curva_corte.loc[curva_corte["corte"].between(5.7, 6.3)].round(3))
print("Corte congelado antes do teste:", CORTE_ESCOLHIDO)
print(f"No desenvolvimento: sensibilidade {linha_corte['sensibilidade']:.1%} e precisão {linha_corte['precisão']:.1%}.")

fig, ax = plt.subplots(figsize=(7.5, 4.0), constrained_layout=True)
ax.plot(curva_corte["corte"], curva_corte["sensibilidade"], label="Sensibilidade", color="#2F6B56")
ax.plot(curva_corte["corte"], curva_corte["precisão"], label="Precisão", color="#9B3046")
ax.axvline(CORTE_ESCOLHIDO, color="#B8862B", linestyle="--", label=f"Corte {CORTE_ESCOLHIDO:.1f}")
ax.axhline(0.75, color="#52606D", linestyle=":", label="Sensibilidade mínima")
ax.set(xlabel="Corte na nota prevista", ylabel="Proporção", ylim=(0, 1), title="Escolha da regra de triagem no desenvolvimento")
ax.legend()
fig.savefig(PASTA_FIGURAS / "curva_corte.png", bbox_inches="tight")
plt.show()


### Teste final usado uma vez

Depois de congelar modelo e corte, ajustamos cada alternativa em todo o desenvolvimento e mostramos sua métrica de teste. Esses números servem para análise final; não alteram a escolha. A referência permanece na tabela para dar escala ao ganho.


In [ ]:
linhas_teste = []
previsoes_teste_por_modelo = {}
for nome, modelo in modelos.items():
    modelo.fit(X_desenvolvimento, y_desenvolvimento)
    previsao = modelo.predict(X_teste)
    previsoes_teste_por_modelo[nome] = previsao
    linhas_teste.append(
        {
            "modelo": nome,
            "MAE teste": mean_absolute_error(y_teste, previsao),
            "RMSE teste": mean_squared_error(y_teste, previsao) ** 0.5,
            "R² teste": r2_score(y_teste, previsao),
        }
    )

comparacao_teste = pd.DataFrame(linhas_teste)
comparacao_final = comparacao_validacao.merge(comparacao_teste, on="modelo")
display(
    comparacao_final[["modelo", "MAE VC", "pior dobra", "melhor dobra", "MAE teste", "RMSE teste", "R² teste"]]
    .sort_values("MAE VC").round(3)
)

previsao_teste = previsoes_teste_por_modelo[MODELO_ESCOLHIDO]
selecao_teste = previsao_teste >= CORTE_ESCOLHIDO
interesse_teste = y_teste.to_numpy() >= 7
metricas_operacionais = {
    "sensibilidade": recall_score(interesse_teste, selecao_teste),
    "precisão": precision_score(interesse_teste, selecao_teste),
    "F1": f1_score(interesse_teste, selecao_teste),
    "selecionados": int(selecao_teste.sum()),
    "total": int(len(y_teste)),
    "bons_recuperados": int((selecao_teste & interesse_teste).sum()),
    "bons_total": int(interesse_teste.sum()),
}
display(pd.DataFrame([metricas_operacionais]).round(3))


## Passo 6 — Analisar erros, dependência das variáveis e vazamento

O erro global esconde a regressão para a média. Por isso, examinamos MAE por nota e por tipo, sempre com o tamanho de cada grupo. A importância por permutação mede quanto o MAE aumenta quando embaralhamos uma entrada no teste; ela indica dependência preditiva, não causalidade.

**Auditoria de vazamento:** `quality` não entra em `X`; duplicatas completas saem antes da divisão; as transformações aprendem dentro de cada dobra; o teste não participa da escolha; todas as entradas físico-químicas e `wine_type` estão disponíveis antes da decisão de triagem.


In [ ]:
erros = pd.DataFrame(
    {
        "wine_type": X_teste["wine_type"].to_numpy(),
        "nota_real": y_teste.to_numpy(),
        "nota_prevista": previsao_teste,
    },
    index=X_teste.index,
)
erros["erro_absoluto"] = (erros["nota_real"] - erros["nota_prevista"]).abs()

erros_por_nota = erros.groupby("nota_real").agg(n=("erro_absoluto", "size"), MAE=("erro_absoluto", "mean"))
erros_por_tipo = erros.groupby("wine_type").agg(n=("erro_absoluto", "size"), MAE=("erro_absoluto", "mean"))
display(erros_por_nota.round(3))
display(erros_por_tipo.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), constrained_layout=True)
axes[0].bar(erros_por_nota.index.astype(str), erros_por_nota["MAE"], color="#7D263C")
axes[0].set(title="Erro por nota no teste", xlabel="Nota real", ylabel="MAE")
for x, (mae, n) in enumerate(zip(erros_por_nota["MAE"], erros_por_nota["n"])):
    axes[0].text(x, mae + 0.05, f"n={n}", ha="center", fontsize=8)
axes[1].bar(["Tinto", "Branco"], erros_por_tipo.loc[["red", "white"], "MAE"], color=["#7D263C", "#B89945"])
axes[1].set(title="Erro por tipo no teste", ylabel="MAE")
for x, tipo in enumerate(["red", "white"]):
    axes[1].text(x, erros_por_tipo.loc[tipo, "MAE"] + 0.02, f"n={erros_por_tipo.loc[tipo, 'n']}", ha="center", fontsize=8)
fig.savefig(PASTA_FIGURAS / "erros_grupos.png", bbox_inches="tight")
plt.show()


In [ ]:
resultado_importancia = permutation_importance(
    modelo_selecionado,
    X_teste,
    y_teste,
    scoring="neg_mean_absolute_error",
    n_repeats=20,
    random_state=SEMENTE,
    n_jobs=1,
)
importancia = pd.DataFrame(
    {
        "variavel": X_teste.columns,
        "aumento_MAE": resultado_importancia.importances_mean,
        "desvio": resultado_importancia.importances_std,
    }
).sort_values("aumento_MAE", ascending=False)
display(importancia.round(3))

top = importancia.head(7).sort_values("aumento_MAE")
fig, ax = plt.subplots(figsize=(7.5, 4.1), constrained_layout=True)
ax.barh(top["variavel"], top["aumento_MAE"], xerr=top["desvio"], color="#9B3046", alpha=0.9)
ax.set(xlabel="Aumento do MAE ao embaralhar a variável", title="Importância por permutação no teste")
fig.savefig(PASTA_FIGURAS / "importancia_permutacao.png", bbox_inches="tight")
plt.show()


### Uso, limitações e restrições

**Decisão apoiada.** O modelo ordena amostras para degustação prioritária. Com o corte escolhido no desenvolvimento, uma amostra segue para o painel quando `nota_prevista >= 6,0`. O custo aceito é enviar falsos positivos para reduzir a perda de vinhos realmente avaliados com nota 7 ou maior.

**Limitações.** A base concentra notas 5 e 6 e contém poucos exemplos extremos. A nota vem de avaliação sensorial, e a base não inclui safra, produtor, variedade de uva, preço, armazenamento ou identificador de amostra. O desempenho varia entre as dobras, por nota e por tipo. A remoção de duplicatas pode retirar observações legítimas.

**Restrições.** O modelo não substitui degustadores, não aprova ou rejeita lotes automaticamente, não estima segurança sanitária ou preço e não deve ser transferido para outras regiões, variedades, laboratórios ou períodos sem validação externa. A importância das variáveis não autoriza intervenções químicas.


## Respostas rápidas para o banco de perguntas

1. **Referência e ganho:** a referência prevê a mediana do desenvolvimento. Compare seu MAE de teste com o da floresta na tabela final e calcule a redução relativa.
2. **Saída:** o modelo devolve uma nota numérica. Ela responde à pergunta de regressão e ordena amostras.
3. **Decisão:** amostras com previsão de pelo menos 6,0 seguem primeiro para degustação humana.
4. **Métrica:** usamos MAE em pontos de nota. RMSE e R² complementam a leitura.
5. **Erro relevante:** deixar passar um vinho realmente 7 ou maior prejudica a triagem; selecionar um falso positivo consome degustação.
6. **Variáveis:** álcool, acidez volátil e dióxido de enxofre livre aparecem no topo da importância por permutação desta execução.
7. **Generalização:** as mesmas dobras mostram o intervalo de MAE; o teste permaneceu isolado até o final.
8. **Descarte:** removemos 1.177 duplicatas completas; não removemos colunas preditoras.
9. **Informação posterior:** nenhuma entrada depende da nota futura. `quality` fica fora de `X`.
10. **Custo da complexidade:** a floresta reduz o MAE, mas perde a explicação por um único conjunto de coeficientes ou caminho.
11. **Outra divisão:** o resultado mudaria. A dispersão das cinco dobras quantifica parte dessa sensibilidade.
12. **Proibição:** não usar para substituir o painel humano, certificar segurança, definir preço ou extrapolar sem validação.
13. **Modelos testados:** referência, regressão linear, k-NN, árvore e floresta. A escolha usou apenas validação cruzada; o teste veio depois.


## Aplicar o modelo a uma nova amostra

A função abaixo exige todas as entradas e devolve a nota prevista e a decisão de triagem. A amostra de demonstração vem do conjunto de teste apenas para mostrar a interface; não representa implantação externa.


In [ ]:
def prever_amostra(amostra):
    tabela = pd.DataFrame([amostra], columns=X.columns)
    nota_prevista = float(modelo_selecionado.predict(tabela)[0])
    return {
        "nota_prevista": round(nota_prevista, 3),
        "enviar_para_degustacao_prioritaria": bool(nota_prevista >= CORTE_ESCOLHIDO),
    }

exemplo = X_teste.iloc[0].to_dict()
print(prever_amostra(exemplo))


## Registro dos resultados e checklist final

Os arquivos abaixo guardam as tabelas e figuras usadas no relatório e na apresentação. Eles resultam desta execução e não substituem o notebook.


In [ ]:
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
curva_arvore.to_csv(PASTA_RESULTADOS / "curva_arvore.csv", index=False)
tabela_ajustes.to_csv(PASTA_RESULTADOS / "ajustes_modelos.csv", index=False)
comparacao_validacao.to_csv(PASTA_RESULTADOS / "comparacao_validacao.csv", index=False)
comparacao_teste.to_csv(PASTA_RESULTADOS / "comparacao_teste.csv", index=False)
curva_corte.to_csv(PASTA_RESULTADOS / "curva_corte.csv", index=False)
erros_por_nota.to_csv(PASTA_RESULTADOS / "erros_por_nota.csv")
erros_por_tipo.to_csv(PASTA_RESULTADOS / "erros_por_tipo.csv")
importancia.to_csv(PASTA_RESULTADOS / "importancia_permutacao.csv", index=False)

linha_final = comparacao_teste.set_index("modelo").loc[MODELO_ESCOLHIDO]
linha_referencia = comparacao_teste.set_index("modelo").loc["Referência (mediana)"]
resumo = {
    "data_execucao": "2026-09-29",
    "random_state": SEMENTE,
    "linhas_originais": int(len(dados_brutos)),
    "duplicatas_removidas": int(len(dados_brutos) - len(base)),
    "linhas_limpas": int(len(base)),
    "linhas_desenvolvimento": int(len(X_desenvolvimento)),
    "linhas_teste": int(len(X_teste)),
    "modelo_escolhido": MODELO_ESCOLHIDO,
    "profundidade_arvore": profundidade_escolhida,
    "corte": CORTE_ESCOLHIDO,
    "mae_teste_modelo": float(linha_final["MAE teste"]),
    "rmse_teste_modelo": float(linha_final["RMSE teste"]),
    "r2_teste_modelo": float(linha_final["R² teste"]),
    "mae_teste_referencia": float(linha_referencia["MAE teste"]),
    "reducao_mae_referencia": float(1 - linha_final["MAE teste"] / linha_referencia["MAE teste"]),
    **{f"triagem_{k}": float(v) if isinstance(v, (float, np.floating)) else int(v) for k, v in metricas_operacionais.items()},
}
(PASTA_RESULTADOS / "resumo_metricas.json").write_text(
    json.dumps(resumo, ensure_ascii=False, indent=2), encoding="utf-8"
)

display(pd.DataFrame([resumo]).T.rename(columns={0: "resultado"}))
print("\nChecklist:")
print("[x] Dados auditados e duplicatas tratadas com justificativa")
print("[x] Teste reservado antes das escolhas")
print("[x] Referência e métrica definidas")
print("[x] Árvore ajustada e sobreajuste identificado")
print("[x] Modelos comparados nas mesmas dobras")
print("[x] Modelo e corte congelados antes do teste")
print("[x] Teste final, erros por grupo e importância analisados")
print("[x] Uso, limitações, restrições e perguntas da banca registrados")
